# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2


In [6]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

170498071/170498071 [==============================] - 1812s 11us/step
X_train shape: (50000, 32, 32, 3)
X_test shape: (10000, 32, 32, 3)


## Definiciones de modelos

In [10]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")


def build_cnn_simple():
    inputs = layers.Input(shape=(32, 32, 3))
    x = data_augmentation(inputs)
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    return models.Model(inputs, outputs, name="CNN_Simple")


def build_cnn_regularized():
    inputs = layers.Input(shape=(32, 32, 3))
    x = data_augmentation(inputs)

    for filters, dropout in [(32, 0.2), (64, 0.3), (128, 0.4)]:
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(dropout)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    return models.Model(inputs, outputs, name="CNN_Regularized")


def build_transfer_learning():
    inputs = layers.Input(shape=(32, 32, 3))
    x = data_augmentation(inputs)
    x = layers.Resizing(96, 96)(x)

    # MobileNetV2 espera valores en el rango [-1, 1].
    x = layers.Rescaling(2.0, offset=-1.0)(x)

    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(96, 96, 3)
    )
    base_model.trainable = False

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    return models.Model(
        inputs,
        outputs,
        name="Transfer_Learning_MobileNetV2"
    )


model_simple = build_cnn_simple()
model_reg = build_cnn_regularized()
model_tl = build_transfer_learning()

2026-09-24 20:10:43.483121: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-09-24 20:10:43.485559: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


9406464/9406464 [==============================] - 4s 0us/step


## Entrenamiento de modelos.

In [11]:
# Definición de callbacks
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

models_list = [model_simple, model_reg, model_tl]
histories = {}

# Parámetros de entrenamiento
EPOCHS = 15
BATCH_SIZE = 64

for model in models_list:
    print(f"\n================ Entrenando: {model.name} ================")
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )
    histories[model.name] = history


================ Entrenando: CNN_Simple ================
Epoch 1/15


2026-09-24 20:11:33.915913: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 614400000 exceeds 10% of free system memory.


782/782 [==============================] - 14s 17ms/step - loss: 1.7666 - accuracy: 0.3576 - val_loss: 1.3988 - val_accuracy: 0.4890
Epoch 2/15
782/782 [==============================] - 12s 16ms/step - loss: 1.5267 - accuracy: 0.4516 - val_loss: 1.2866 - val_accuracy: 0.5294
Epoch 3/15
782/782 [==============================] - 12s 16ms/step - loss: 1.4296 - accuracy: 0.4853 - val_loss: 1.1934 - val_accuracy: 0.5738
Epoch 4/15
782/782 [==============================] - 12s 16ms/step - loss: 1.3743 - accuracy: 0.5101 - val_loss: 1.1529 - val_accuracy: 0.5902
Epoch 5/15
782/782 [==============================] - 12s 16ms/step - loss: 1.3276 - accuracy: 0.5264 - val_loss: 1.1278 - val_accuracy: 0.5993
Epoch 6/15
782/782 [==============================] - 12s 16ms/step - loss: 1.2881 - accuracy: 0.5419 - val_loss: 1.0793 - val_accuracy: 0.6225
Epoch 7/15
782/782 [==============================] - 12s 16ms/step - loss: 1.2580 - accuracy: 0.5537 - val_loss: 1.0526 - val_accuracy: 0.6291
Epo

2026-09-24 20:14:46.837276: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 614400000 exceeds 10% of free system memory.


782/782 [==============================] - 26s 32ms/step - loss: 1.9146 - accuracy: 0.3176 - val_loss: 2.1419 - val_accuracy: 0.2832
Epoch 2/15
782/782 [==============================] - 24s 30ms/step - loss: 1.5756 - accuracy: 0.4235 - val_loss: 1.7699 - val_accuracy: 0.3622
Epoch 3/15
782/782 [==============================] - 24s 31ms/step - loss: 1.4787 - accuracy: 0.4622 - val_loss: 1.8352 - val_accuracy: 0.3975
Epoch 4/15
782/782 [==============================] - 24s 30ms/step - loss: 1.4122 - accuracy: 0.4867 - val_loss: 1.7502 - val_accuracy: 0.4121
Epoch 5/15
782/782 [==============================] - 23s 29ms/step - loss: 1.3573 - accuracy: 0.5111 - val_loss: 1.6840 - val_accuracy: 0.4181
Epoch 6/15
782/782 [==============================] - 23s 29ms/step - loss: 1.3174 - accuracy: 0.5254 - val_loss: 1.4344 - val_accuracy: 0.4845
Epoch 7/15
782/782 [==============================] - 23s 29ms/step - loss: 1.2765 - accuracy: 0.5431 - val_loss: 1.6922 - val_accuracy: 0.4300
Epo

2026-09-24 20:20:40.913446: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 614400000 exceeds 10% of free system memory.


782/782 [==============================] - 73s 91ms/step - loss: 1.1455 - accuracy: 0.6064 - val_loss: 0.5729 - val_accuracy: 0.8050
Epoch 2/15
782/782 [==============================] - 70s 89ms/step - loss: 0.9197 - accuracy: 0.6819 - val_loss: 0.5342 - val_accuracy: 0.8173
Epoch 3/15
782/782 [==============================] - 66s 85ms/step - loss: 0.8913 - accuracy: 0.6901 - val_loss: 0.5378 - val_accuracy: 0.8158
Epoch 4/15
782/782 [==============================] - 68s 87ms/step - loss: 0.8929 - accuracy: 0.6919 - val_loss: 0.5115 - val_accuracy: 0.8227
Epoch 5/15
782/782 [==============================] - 70s 89ms/step - loss: 0.8813 - accuracy: 0.6940 - val_loss: 0.5166 - val_accuracy: 0.8267
Epoch 6/15
782/782 [==============================] - 73s 94ms/step - loss: 0.8766 - accuracy: 0.6956 - val_loss: 0.5447 - val_accuracy: 0.8165
Epoch 7/15
782/782 [==============================] - 75s 95ms/step - loss: 0.8805 - accuracy: 0.6937 - val_loss: 0.5299 - val_accuracy: 0.8220
Epo

## Estadística y gráficos

In [12]:
plt.figure(figsize=(14, 6))

# Gráfico de Precisión (Accuracy)
plt.subplot(1, 2, 1)
for name, history in histories.items():
    plt.plot(history.history['val_accuracy'], label=f'{name} (Val)')
plt.title('Comparación de Accuracy en Validación')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Gráfico de Pérdida (Loss)
plt.subplot(1, 2, 2)
for name, history in histories.items():
    plt.plot(history.history['val_loss'], label=f'{name} (Val)')
plt.title('Comparación de Loss en Validación')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Resumen en tabla comparativa de métricas finales
results = []
for model in models_list:
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    results.append({"Modelo": model.name, "Test Loss": round(loss, 4), "Test Accuracy": round(acc, 4)})

df_results = pd.DataFrame(results)
print("\n--- RESUMEN FINAL DE EVALUACIÓN EN TEST ---")
print(df_results.to_string(index=False))

<Figure size 1400x600 with 2 Axes>


--- RESUMEN FINAL DE EVALUACIÓN EN TEST ---
                       Modelo  Test Loss  Test Accuracy
                   CNN_Simple     1.0082         0.6525
              CNN_Regularized     1.1689         0.5943
Transfer_Learning_MobileNetV2     0.5115         0.8227


# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejor? ¿Por qué? ¿Qué mejoraría? ¿Cómo lo mejoraría?

### ¿Cuál fue el mejor modelo? y ¿Por qué?
El modelo **Transfer_Learning_MobileNetV2** superó a los demás entrenados desde cero, alcanzando una precisión de **82.27%** y una pérdida de **0.5115**.

Las razones se deben a que **Transfer Learning** utiliza pesos preentrenados en ImageNet; la red ya cuenta con los parámetros y variables complejas (bordes, texturas, formas, etc.) aprendidas sin necesidad de calcularlas desde cero. Esto generaliza mucho mejor en el conjunto de prueba.

Mientras que con CNN Simple y CNN Regularizada:
    * En CNN Simple, superó a la regularizada con un puntaje de **65.25%** de precisión, pero igualmente se tardó en procesar todo, ya que tuvo que aprender desde cero.

    * En contraste con el puntaje de **59.43%** de su variante regularizada, tomando en cuenta también su **Test Loss: 1.1689**, seguramente haya un problema de subajuste o una penalización excesiva de múltiples capas "Dropout" y "BatchNormalization" para pocas épocas de entrenamiento.

### ¿Qué hay que mejorar?
1. **Regularizar de forma moderada en CNN propia:** Se podría reducir la tasa de *Dropout* en 'CNN Regularizada' (por ejemplo, a valores entre 0.1 y 0.2) para evitar que pierda demasiada información.

2. **Aumentar las épocas:** Entrenar la regularizada durante más épocas, debido a que *BatchNormalization* y *Dropout* suelen requerir más iteraciones para converger.

3. **Fine-Tuning en MobileNetV2:** Descongelar las últimas capas de la base y entrenar a una tasa de aprendizaje muy baja ("1e-5") para adaptar características específicas al conjunto.

### ¿Cómo lo mejoraría?
1. Configurar un ajuste fino en MobileNetV2 después del entrenamiento inicial congelado.

2. Aplicar una búsqueda de hiperparámetros (*Grid Search* o *KerasTuner*) en "CNN_Regularizada" para optimizar la tasa de aprendizaje del optimizador ADAM y la proporción de *Dropout*.
